# 🔀 ColumnTransformer — Quick Notes
---
> **Dataset:** `tips` (seaborn) + `titanic` | **Library:** `sklearn.compose`

## 📌 Key Points
- Applies **different transformers to different columns** in one step
- Handles numeric and categorical columns **separately & simultaneously**
- Output is a single numpy array (or DataFrame)
- ✅ Fully compatible with sklearn `Pipeline`
- `remainder='passthrough'` → keep untouched columns
- `remainder='drop'` → drop untouched columns (default)

## 🔑 Structure
```python
ColumnTransformer([
    ('name1', Transformer1(), ['col1', 'col2']),  # numeric
    ('name2', Transformer2(), ['col3', 'col4']),  # categorical
], remainder='passthrough')
```

## 📊 Dataset: `tips`
| Column Type | Columns | Transformer |
|---|---|---|
| Numeric | `tip`, `size` | `StandardScaler` |
| Categorical | `sex`, `smoker`, `day`, `time` | `OneHotEncoder` |

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

df = sns.load_dataset('tips')
x = df.drop(columns=['total_bill'])
y = df['total_bill']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("x_train shape:", x_train.shape)
print("Columns:", list(x_train.columns))
print(x_train.head(3))

In [ ]:
# Define column groups
num_cols = ['tip', 'size']
cat_cols = ['sex', 'smoker', 'day', 'time']

# Build ColumnTransformer
ct = ColumnTransformer([
    ('scaler', StandardScaler(),                                    num_cols),
    ('ohe',    OneHotEncoder(drop='first', sparse_output=False),   cat_cols)
], remainder='passthrough')

# Fit on train, transform both
x_train_ct = ct.fit_transform(x_train)
x_test_ct  = ct.transform(x_test)

print("Shape before ColumnTransformer:", x_train.shape)
print("Shape after  ColumnTransformer:", x_train_ct.shape)

In [ ]:
# Get output column names
num_names = num_cols
cat_names = list(ct.named_transformers_['ohe'].get_feature_names_out(cat_cols))
all_names = num_names + cat_names
print("Output columns:", all_names)

df_ct = pd.DataFrame(x_train_ct, columns=all_names)
df_ct.head()

In [ ]:
# Use with a model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression()
lr.fit(x_train_ct, y_train)
y_pred = lr.predict(x_test_ct)
print("R2 Score with ColumnTransformer:", round(r2_score(y_test, y_pred), 4))

In [ ]:
# Titanic example — real-world ColumnTransformer
df_t = pd.read_csv("/content/titanic - titanic.csv")
df_t = df_t.drop(columns=['Cabin','Name','Ticket','PassengerId'], errors='ignore')
df_t['Age'].fillna(df_t['Age'].median(), inplace=True)
df_t['Embarked'].fillna(df_t['Embarked'].mode()[0], inplace=True)
df_t.dropna(inplace=True)

x_t = df_t.drop(columns=['Survived'])
y_t = df_t['Survived']

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols_t = ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass']
cat_cols_t = ['Sex', 'Embarked']

# ColumnTransformer with sub-pipelines per column type
ct_t = ColumnTransformer([
    ('num', StandardScaler(), num_cols_t),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols_t)
])

x_train_t, x_test_t, y_train_t, y_test_t = train_test_split(
    x_t[num_cols_t + cat_cols_t], y_t, test_size=0.2, random_state=42)

x_train_enc = ct_t.fit_transform(x_train_t)
x_test_enc  = ct_t.transform(x_test_t)
print("Titanic after ColumnTransformer:", x_train_enc.shape)

## 🔧 Key Parameters
| Param | Meaning |
|---|---|
| `transformers` | List of (name, transformer, columns) tuples |
| `remainder` | `'drop'` or `'passthrough'` for other cols |
| `verbose_feature_names_out` | Include transformer name in output cols |

## 🥇 Remember
- `ct.named_transformers_['name']` → access individual fitted transformer
- `ct.get_feature_names_out()` → all output column names
- ✅ Always used inside a `Pipeline` in production
- Fit on train only — same golden rule applies